# Lab21 — GSM8K B2/B3 on Kaggle

Upload `Lab21_Kaggle_GSM8K.zip` as a **private Dataset**, add it as Input, enable **Internet** and select an available GPU (T4 x2 recommended; the code uses GPU 0 only).

Run cells in order. This is a separate Kaggle math run; existing Colab CSKH and math files on Drive are untouched. Baselines are measured before training in the Kaggle software/GPU environment. Input files are copied from read-only `/kaggle/input` into writable `/kaggle/working`.

The corpus has 250 train, 50 val, 50 eval, and 15 regression probes. Both masks use the same model revision, data, rank16, alpha32, LR1e-4 and 32 steps. Valid traces require a nonempty closed thinking block; the known opening tag in the generation prompt is accounted for.

Results/adapter exports are in `/kaggle/working`. Save a version **with output files** and download the ZIP before ending the session. Runtime storage alone is not a permanent backup. No full merged model is exported.

Public GSM8K may have been in Qwen pretraining. This is a learner-curated format/mask experiment, not evidence of novel private data or a full official GSM8K benchmark score.


In [1]:
# 1. Load private input and prepare writable experiment folder
import os, sys, shutil, json, zipfile, subprocess, re
from pathlib import Path
# Use one GPU to keep effective batch=16, including on a T4 x2 runtime.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HOME'] = '/tmp/lab21_hf_cache'
WORK = Path('/kaggle/working/lab21_kaggle')
INPUT = Path('/kaggle/input')
if not (WORK / 'run_math.py').exists():
    archives = list(INPUT.rglob('Lab21_Kaggle_GSM8K.zip'))
    if archives:
        with zipfile.ZipFile(archives[0]) as z:
            assert all(name.startswith('lab21_kaggle/') for name in z.namelist())
            z.extractall('/kaggle/working')
    else:
        candidates = [p.parent for p in INPUT.rglob('run_math.py')
                      if (p.parent / 'src/labkit').is_dir()
                      and (p.parent / 'bonus_gsm8k/results/dataset_manifest.json').exists()]
        assert candidates, 'Add Input: upload Lab21_Kaggle_GSM8K.zip as a private dataset first.'
        shutil.copytree(candidates[0], WORK)
    for p in WORK.rglob('*'):
        p.chmod(0o755 if p.is_dir() else 0o644)
MATH_DIR = WORK / 'bonus_gsm8k'
os.chdir(WORK)
manifest = json.loads((MATH_DIR / 'results/dataset_manifest.json').read_text())
print('Prepared dataset:', manifest['sizes'])
print('Writable results:', MATH_DIR / 'results')


Prepared dataset: {'train': 250, 'val': 50, 'eval_target': 50, 'eval_regression': 15}
Writable results: /kaggle/working/lab21_kaggle/bonus_gsm8k/results


In [2]:
# 2. Dependencies — keep Kaggle's working CUDA-enabled torch
from importlib.metadata import version
import torch
assert torch.cuda.is_available(), 'Settings > Accelerator: select GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__, 'CUDA:', torch.version.cuda)
requirements = []
for line in (WORK / 'requirements.txt').read_text().splitlines():
    name = re.match(r'^\s*([A-Za-z0-9_.-]+)', line)
    # B2/B3 use 16-bit LoRA, not the optional 4-bit contrast.
    if name and name[1].lower() not in {'torch', 'bitsandbytes'}:
        requirements.append(line)
req = WORK / 'requirements-kaggle.txt'
req.write_text('\n'.join(requirements) + '\n')
constraints = WORK / 'torch-constraint.txt'
constraints.write_text('torch==' + version('torch') + '\n')
print('Installing training libraries; preserving existing torch...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req),
                '-c', str(constraints)], check=True)
print('Dependencies ready.')


GPU: Tesla T4
torch: 2.11.0+cu128 CUDA: 12.8
Installing training libraries; preserving existing torch...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 927.2/927.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 43.8 MB/s eta 0:00:00
Dependencies ready.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.


In [3]:
# 3. Freeze Kaggle math baselines BEFORE training

def run_visible(stage):
    command = [sys.executable, '-u', str(WORK / 'run_math.py'),
               '--home', str(MATH_DIR), '--stage', stage]
    with (WORK / (stage + '.log')).open('a') as log:
        p = subprocess.Popen(command, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        if p.wait() != 0:
            raise RuntimeError('Stage failed: ' + stage + '. Read the log above.')

run_visible('baseline')


Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so

Fetching 2 files: 100%|██████████| 2/2 [00:47<00:00, 23.61s/it]

Loading weights: 100%|██████████| 426/426 [00:04<00:00, 86.52it/s]
  [base-naive] batch 1/25     38s elapsed  ~  912s left
  [base-naive] batch 2/25     74s elapsed  ~  851s left
  [base-naive] batch 3/25    110s elapsed  ~  807s left
  [base-naive] batch 4/25    146s elapsed  ~  767s left
  [base-naive] batch 5/25    182s elapsed  ~  728s left
  [base-naive] batch 6/25    218s elapsed  ~  691s left
  [base-naive] batch 7/25    254s elapsed  ~  653s left
  [base-naive] batch 8/25    290s elapsed  ~  617s left
  [base-n

In [5]:
# 4. Train/evaluate the two matched mask runs, then summarize
run_visible('all')


Baselines already frozen; reusing them.
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 915.49it/s]

Loading weights: 100%|██████████| 426/426 [00:04<00:00, 89.62it/s] 

Truncating train dataset: 100%|██████████| 250/250 [00:00<00:00, 9010.63 examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/250 [00:00<?, ? examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 250/250 [00:00<00:00, 31652.26 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The mo

In [6]:
# 5. Read the summary and create a small downloadable result ZIP
summary = MATH_DIR / 'results/math_bonus_summary.json'
print(summary.read_text())
archive = Path('/kaggle/working/Lab21_Kaggle_BonusResults.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ('data', 'results', 'adapters', 'scripts'):
        for p in (MATH_DIR / folder).rglob('*'):
            if p.is_file() and not any(part.startswith('checkpoint-') for part in p.parts):
                z.write(p, Path('bonus_gsm8k') / p.relative_to(MATH_DIR))
    z.write(WORK / 'run_math.py', 'run_math.py')
print('Result archive:', archive)
print('Save Version with output files, then download the result ZIP from Output.')


{
  "configuration": {
    "model": "unsloth/Qwen3.5-4B",
    "model_revision": "3764fa359b9082ea5a1e4a5e3ac3aaf6e9671636",
    "data_checksums": {
      "train.jsonl": "b00b575ae5db789c6805f8e63780efde7c300ad9a115ebcceb36d686ca130c75",
      "val.jsonl": "ed87c44b7a172affb623531249e259699918c2a210edbd8995f676ec7d8fe009",
      "eval_target.jsonl": "c3616b38426164ece9498c77d7a0582ed248077711b335d44ac796a38eccc30f",
      "eval_regression.jsonl": "9c6ba90ea86e28fbc7102bf4cc4cb1e23c4668cf95bfcae55c3c114d34dc133b"
    },
    "runner_sha256": "4560e4a646e30b1ad2a9af63c7a82094cf2532957af3c37d7f9d239341d0f854",
    "package_versions": {
      "torch": "2.11.0+cu128",
      "transformers": "5.16.1",
      "trl": "1.14.2",
      "peft": "0.20.0",
      "accelerate": "1.14.0",
      "datasets": "5.1.0",
      "tokenizers": "0.23.1"
    },
    "labkit_hashes": {
      "__init__.py": "915328d68af3de04bdbebba175374005c4859cb9ef543bf8d4e48f6e414a7a68",
      "config.py": "65f34e0261921f94b92693bf61